# Day-search results

Exploratory reporting on a `concopt search` run: the shape of the whole
candidate set (not just its top rows), and a close look at the winning
day's profile.

**Prerequisite** -- `concopt search` only writes its top rows by
default. Run it with `--out-all` first so the full ~31,000-candidate
frame is on disk for this notebook to load:

```
concopt search --pln tests/data/KJFKEGLL_CONC_01.pln \
               --npz data/era5/route_legs.npz \
               --surface-npz <your surface .npz, from era5.reduce_surface_to_npz> \
               --zfw <your ZFW in tonnes>                --out results.csv --out-all results_all.csv
```

TOW is an outcome of `--zfw` (solved per candidate), so it is read back
from the CSV's own `tow_t` column here rather than being an input. If the
search was run with the optional `--tow` override, `tow_override_t` is set.

Everything computed here reuses concopt's own functions (`limits`,
`search.march_legs`, `report.flight_profile`, ...) -- this notebook loads
and plots, it does not reimplement the march. Sections 6-7 also need the
arrival wind files (`--subsonic-npz`, `--arrival-upper-npz`; paths in the
config cell), because `report.flight_profile` runs the full arrival model
for the winning day.

In [ ]:
"""Day-search results: imports and configuration."""

import calendar

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from concopt import report, runways
from concopt.atmos import isa
from concopt.era5 import load_legs_npz
from concopt.params import C_TO_K, MMO, TOTAL_TEMP_MAX_C
from concopt.report import _format_mmss, flight_profile
from concopt.route import build_legs, climb_cruise_segment, parse_pln
from concopt.search import (
    TARGET_FL,
    _format_hmm,
    local_to_departure_utc,
    march_legs,
)

# Same colour for the same series in every cell below.
PALETTE = ["#4C72B0", "#DD8452", "#55A868"]  # blue, orange, green

PLN_PATH = "../tests/data/KJFKEGLL_CONC_01.pln"
NPZ_PATH = "../data/era5/route_legs.npz"
# era5.reduce_to_legs run on the post-BARIX legs; both must share a time axis.
SUBSONIC_NPZ_PATH = "../data/era5/subsonic_legs_trimmed.npz"
ARRIVAL_UPPER_NPZ_PATH = "../data/era5/arrival_upper_legs.npz"
RESULTS_ALL_CSV = (
    "../data/results_all.csv"  # from `concopt search --out-all`, see above
)
# `ATCWaypoint id`s in the .pln -- DECEL_ID must match the --decel the search/npz files were built with.
ACCEL_ID = "LINND"
DECEL_ID = "BARIX"
CRUISE_MACH = MMO
TOP_N = 20  # how many rows count as "the shortlist" below

## 1. Load

In [ ]:
plan = parse_pln(PLN_PATH)
legs = build_legs(plan["waypoints"])
mask = climb_cruise_segment(legs, decel_id=DECEL_ID)
cc_idx = np.flatnonzero(mask)
cc_legs = [legs[i] for i in cc_idx]

df = pd.read_csv(RESULTS_ALL_CSV, parse_dates=["local_date", "departure_utc"])
# "" round-trips through CSV as NaN in an otherwise-string column.
for col in ("jfk_flag", "lhr_flag", "flags"):
    df[col] = df[col].fillna("")

print(
    f"{len(df):,} candidates, {df['local_date'].min().date()} "
    f"to {df['local_date'].max().date()}"
)

# TOW is solved per candidate from ZFW (or one --tow override for every row);
# both come from the search's own CSV, so this notebook never assumes a weight.
ZFW_T = float(df["zfw_t"].iloc[0])
tow_overridden = (
    df["tow_override_t"].notna().any() if "tow_override_t" in df else False
)
print(
    f"ZFW {ZFW_T:.1f} t; "
    f"TOW {df['tow_t'].min():.1f}-{df['tow_t'].max():.1f} t "
    f"(median {df['tow_t'].median():.1f})"
    + (" -- --tow override" if tow_overridden else " -- solved")
)


def _still_air_reference(
    route_legs, route_idx, n_route_legs, tow_t, cruise_mach
):
    """Still-air, ISA+0 reference for the climb+cruise segment: the same
    march_legs the search itself uses, fed a synthetic zero-wind
    atmosphere. FL450-FL600 sits entirely inside the 11-20 km isothermal
    layer (atmos.isa), so one constant temperature reproduces ISA+0 at
    every one of the 4 raw ERA5 pressure levels without needing a
    pressure -> altitude inversion. Returns (climb_time_s, cruise_time_s)."""
    t_iso, _ = isa(TARGET_FL.mean() * 100.0 * 0.3048)
    level = np.array([150.0, 125.0, 100.0, 70.0])  # era5.py's mandatory levels
    time = np.array(["2015-01-01", "2030-01-01"], dtype="datetime64[ns]")
    shape = (len(time), len(level), n_route_legs)
    data = {
        "time": time,
        "level": level,
        "u": np.zeros(shape),
        "v": np.zeros(shape),
        "t": np.full(shape, t_iso),
    }
    ref_dep_i8 = np.array(
        [np.datetime64("2020-01-01T12:00:00", "ns").astype("int64")]
    )
    ref_legs_out, _weight_per_leg, ref_climb = march_legs(
        route_legs, route_idx, data, ref_dep_i8, tow_t, cruise_mach
    )
    climb_time_s = float(ref_climb["time_min"][0]) * 60.0
    return climb_time_s, float(ref_legs_out["accumulated_s"][0]) - climb_time_s


still_air_climb_s, still_air_cruise_s = _still_air_reference(
    cc_legs, cc_idx, len(legs), df["tow_t"].median(), CRUISE_MACH
)
print(
    f"still-air ISA+0 climb: {_format_hmm(still_air_climb_s)}, "
    f"cruise: {_format_hmm(still_air_cruise_s)} "
    f"({still_air_cruise_s / 60.0:.0f} min, at the median TOW) -- "
    "for context against the wind-assisted times below"
)

## 2. Total block time, all candidates

In [ ]:
total_min = df["total_time_s"] / 60.0
top_n = df.nsmallest(TOP_N, "total_time_s")
top_min = top_n["total_time_s"] / 60.0
bins = np.histogram_bin_edges(total_min, bins=60).tolist()

best_min = total_min.min()
median_min = total_min.median()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(
    total_min,
    bins=bins,
    color=PALETTE[0],
    alpha=0.8,
    label=f"all candidates (n={len(df):,})",
)
ax.hist(top_min, bins=bins, color=PALETTE[1], alpha=0.9, label=f"top {TOP_N}")
ax.axvline(best_min, color=PALETTE[1], linestyle="--", linewidth=1)
ax.axvline(median_min, color=PALETTE[0], linestyle="--", linewidth=1)
ax.annotate(
    f"best {_format_hmm(best_min * 60.0)}",
    xy=(best_min, ax.get_ylim()[1]),
    xytext=(4, -12),
    textcoords="offset points",
    color=PALETTE[1],
)
ax.annotate(
    f"median {_format_hmm(median_min * 60.0)}",
    xy=(median_min, ax.get_ylim()[1]),
    xytext=(4, -28),
    textcoords="offset points",
    color=PALETTE[0],
)
ax.set_xlabel("total block time (min)")
ax.set_ylabel("candidates")
ax.set_title("Total block time, all candidates")
ax.legend()
fig.tight_layout()

## 3. Total time by month

In [ ]:
month = df["local_date"].dt.month
by_month = [
    df.loc[month == m, "total_time_s"].to_numpy() / 60.0 for m in range(1, 13)
]

fig, ax = plt.subplots(figsize=(9, 4.5))
bp = ax.boxplot(
    by_month,
    tick_labels=[calendar.month_abbr[m] for m in range(1, 13)],
    patch_artist=True,
    showfliers=False,
)
for patch in bp["boxes"]:
    patch.set_facecolor(PALETTE[0])
    patch.set_alpha(0.6)
ax.set_ylabel("total block time (min)")
ax.set_title("Total block time by month")
fig.tight_layout()

## 4. Total time vs along-track wind

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sc = ax.scatter(
    df["mean_wind_kt"],
    df["total_time_s"] / 60.0,
    c=df["mean_isa_dev_k"],
    cmap="Blues",
    s=10,
    alpha=0.6,
    edgecolors="none",
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("mean ISA deviation (K)")
ax.set_xlabel("mean along-track wind (kt)")
ax.set_ylabel("total block time (min)")
ax.set_title("Total time vs along-track wind, coloured by ISA deviation")
fig.tight_layout()

## 5. Top 10

In [ ]:
top10 = df.nsmallest(10, "total_time_s")
top10_display = pd.DataFrame(
    {
        "date": top10["local_date"].dt.strftime("%Y-%m-%d"),
        "local_departure": top10["local_hour"].map(lambda h: f"{h:02d}:00"),
        "total_time": top10["total_time_s"].map(_format_hmm),
        "supersonic_time": top10["supersonic_time_s"].map(_format_hmm),
        "mean_fl": top10["mean_fl"].round(0).astype(int),
        "mean_wind_kt": top10["mean_wind_kt"].round(1),
        "runways": top10["jfk_runway"] + " / " + top10["lhr_runway"],
        "flags": top10["flags"].replace("", "-"),
    }
)
top10_display.reset_index(drop=True)

## 6. Winning day: full-flight profile

`report.flight_profile` runs the winning candidate through the same model
`concopt report` uses (climb table, cruise march, arrival model) and lays the
result out from brake release to touchdown. The three regions have different
resolution, and the plots below say so:

- **Climb** (brake release to FL502): `conc_climb.csv`'s 18 cumulative levels
  give altitude, time, distance and fuel. The table has **no speeds**, so
  climb Mach/CAS/TAS are left blank and only the whole-climb mean ground
  speed is drawn (dashed).
- **Cruise**: the per-sub-leg march, as before.
- **Arrival** (BARIX to touchdown): the four segment totals from
  `arrival.arrival()`, straight-line subdivided; Mach/CAS/TAS derived from
  the schedule (decel M2.0 to M1.0, level M0.95, descent at the 325/350/380
  kt CAS schedule). The approach allowance has no speed model.

Takeoff roll and landing are not modelled: the climb table starts at brake
release and the arrival ends at touchdown. The two runway penalties (time
only) appear in the phase table.

If the profile's total differs from the search CSV's `total_time_s` by more
than a minute, the CSV was produced with different inputs (another
`--cruise-mach`, wind files or code version) than this notebook is using.

In [ ]:
winner = df.nsmallest(1, "total_time_s").iloc[0]
winner_date = winner["local_date"].date()
winner_hour = int(winner["local_hour"])
winner_tow_t = float(winner["tow_t"])  # the TOW the search flew this day at
print(
    f"winner TOW {winner_tow_t:.1f} t = ZFW {ZFW_T:.1f} t + "
    f"{winner_tow_t - ZFW_T:.1f} t fuel"
)

npz_data = load_legs_npz(NPZ_PATH)
departure_utc = local_to_departure_utc(winner_date, winner_hour)
dep_i8 = np.array([pd.Timestamp(departure_utc).value], dtype="int64")

# Cruise-only march, kept for the total-temperature cell (6c): the flight
# profile below carries everything else.
legs_out, weight_per_leg, climb = march_legs(
    cc_legs, cc_idx, npz_data, dep_i8, winner_tow_t, CRUISE_MACH
)
full_leg = {
    k: v[0]
    for k, v in legs_out.items()
    if k not in ("accumulated_s", "weight_at_barix")
}
weight_per_leg = weight_per_leg[0]  # weight at the start of each sub-leg

# Cruise-only sub-legs (eff_dist_nm > 0) -- climb-consumed ones carry a
# climb-altitude chosen_fl/wind/etc that was never actually flown at
# cruise (see search.py's eff_dist_nm), so they're dropped.
cruise_mask = full_leg["eff_dist_nm"] > 0.0
ss_legs = [lg for lg, m in zip(cc_legs, cruise_mask) if m]
leg = {k: v[cruise_mask] for k, v in full_leg.items()}
weight_per_leg = weight_per_leg[cruise_mask]
start_cum_nm = np.array(
    [lg.cum_nm - lg.dist_nm for lg in ss_legs]
)  # each sub-leg's own start
temp_k_at_best = leg["temp_c"] + C_TO_K
schedule = report._step_climb_schedule(  # pylint: disable=protected-access
    ss_legs, leg["chosen_fl"]
)  # [(cum_nm, FL), ...] at each step

# The full flight. TOW is re-solved from ZFW exactly as the search did, unless
# that search used the --tow override.
tow_override = (
    winner["tow_override_t"]
    if "tow_override_t" in winner and pd.notna(winner["tow_override_t"])
    else None
)
fp = flight_profile(
    PLN_PATH,
    NPZ_PATH,
    winner_date,
    winner_hour,
    ZFW_T,
    ACCEL_ID,
    DECEL_ID,
    tow_t=tow_override,
    subsonic_npz_path=SUBSONIC_NPZ_PATH,
    arrival_upper_npz_path=ARRIVAL_UPPER_NPZ_PATH,
    cruise_mach=CRUISE_MACH,
    runway_penalties_s=(winner["jfk_penalty_s"], winner["lhr_penalty_s"]),
)
profile, phases, summary = fp.profile, fp.phases, fp.summary

gap_s = summary["total_time_s"] - winner["total_time_s"]
print(
    f"profile total {_format_hmm(summary['total_time_s'])} vs search CSV "
    f"{_format_hmm(winner['total_time_s'])} ({gap_s:+.0f} s), "
    f"TOW {summary['tow_t']:.1f} t vs CSV {winner_tow_t:.1f} t"
)
if abs(gap_s) > 60.0:
    print(
        "!! more than a minute apart -- results_all.csv came from different "
        "inputs than this notebook "
        "(cruise Mach, wind files, code version); "
        "re-run `concopt search` before trusting the comparison"
    )

PHASE_COLORS = (
    {  # Okabe-Ito, colour-blind safe; one colour per phase in every plot below
        "climb": "#0072B2",
        "acceleration": "#56B4E9",
        "cruise": "#009E73",
        "deceleration": "#E69F00",
        "subsonic cruise": "#CC79A7",
        "descent": "#D55E00",
        "approach": "#999999",
    }
)
flown = phases[phases["phase"].isin(PHASE_COLORS)].reset_index(drop=True)
phase_legend = [
    Patch(facecolor=PHASE_COLORS[p], label=p) for p in flown["phase"]
]


def plot_by_phase(axes, column, scale=1.0, **kw):
    """One line per phase in that phase's colour; the profile repeats each
    boundary point, so lines join and steps/limit changes are vertical jumps.
    """
    for name, frame in profile.groupby("phase", sort=False):
        axes.plot(
            frame["cum_nm"],
            frame[column] * scale,
            color=PHASE_COLORS[str(name)],
            **kw,
        )


def mark_phase_starts(axes):
    """Light vertical rule at the start of every phase after the first."""
    for start_nm in flown["start_nm"].iloc[1:]:
        axes.axvline(start_nm, color="0.85", linewidth=0.7, zorder=0)


def shade_no_speed_data(axes):
    """Grey out the climb, where conc_climb.csv has no speeds to plot."""
    axes.axvspan(0.0, summary["top_of_climb_nm"], color="0.93", zorder=0)
    axes.annotate(
        "no climb speed data",
        xy=(summary["top_of_climb_nm"] / 2.0, 0.5),
        xycoords=("data", "axes fraction"),
        ha="center",
        fontsize=8,
        color="0.4",
    )


title_tag = f"Winning day {winner_date} {winner_hour:02d}:00"
print(
    f"top of climb at {summary['top_of_climb_nm']:.0f} nm "
    f"(LINND at {summary['linnd_nm']:.0f} nm), "
    f"arrival schedule {summary['schedule_kt']} kt"
)

fig, ax = plt.subplots(figsize=(11, 4.8))
plot_by_phase(ax, "fl", linewidth=2)
cruise_pts = profile[profile["phase"] == "cruise"]
ax.plot(
    cruise_pts["cum_nm"],
    cruise_pts["ceiling_ft"] / 100.0,
    color=PALETTE[1],
    linestyle="--",
    label="ceiling (cruise only)",
)
for step_cum_nm, step_fl in schedule:
    ax.annotate(
        f"FL{step_fl:.0f}",
        xy=(step_cum_nm, step_fl),
        xytext=(0, 6),
        textcoords="offset points",
        fontsize=8,
        ha="center",
    )
mark_phase_starts(ax)
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("flight level")
ax.set_title(
    f"{title_tag} -- flight level vs ceiling, brake release to touchdown"
)
ax.legend(
    handles=phase_legend
    + [
        Line2D(
            [0],
            [0],
            color=PALETTE[1],
            linestyle="--",
            label="ceiling (cruise only)",
        )
    ],
    fontsize=8,
    ncol=2,
    loc="lower center",
)
fig.tight_layout()

The ceiling is the Air France table's altitude attainable at cruise Mach for
the current weight and ISA deviation, so it is drawn over the cruise only --
it is not the limit while climbing or descending. Descent altitude is a
straight line between the segment endpoints (only those are tabulated).

## 6a. Winning day: Mach and Mach limit

Actual Mach (coloured by phase) against the Mach limit (red dashed). The
limit is **M1.0 from brake release to LINND and again from the end of the
decel segment** (subsonic-only stretches); in between it is the lowest of
cruise Mach, the CAS limit and the total-temperature limit. The decel segment
is the transition, so it runs above the M1.0 line until it reaches M1.0. In
cruise, dots are coloured by which limit binds. The climb has no Mach data.

In [ ]:
binding_colors = {
    "cruise_mach": PALETTE[1],
    "CAS": PALETTE[2],
    "total_temp": "#8b4513",  # brown
    "ceiling": "#cccccc",
}
cruise_pts = profile[profile["phase"] == "cruise"]
colors = [binding_colors.get(b, "gray") for b in cruise_pts["binding"]]

fig, ax = plt.subplots(figsize=(11, 4.5))
shade_no_speed_data(ax)
plot_by_phase(ax, "mach", linewidth=2)
ax.scatter(
    cruise_pts["cum_nm"],
    cruise_pts["mach"],
    c=colors,
    s=14,
    alpha=0.8,
    edgecolors="none",
    zorder=3,
)
ax.plot(
    profile["cum_nm"],
    profile["mach_limit"],
    color="red",
    linestyle="--",
    linewidth=1.3,
)
mark_phase_starts(ax)
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("Mach number")
ax.set_title(f"{title_tag} -- Mach actual vs limit")
ax.set_ylim((0.0, 2.2))
ax.legend(
    handles=phase_legend
    + [
        Line2D(
            [0],
            [0],
            color="red",
            linestyle="--",
            linewidth=1.3,
            label="Mach limit",
        ),
        Patch(facecolor=PALETTE[1], label="cruise binding: cruise_mach"),
        Patch(facecolor=PALETTE[2], label="cruise binding: CAS"),
        Patch(facecolor="#8b4513", label="cruise binding: total_temp"),
    ],
    loc="lower right",
    fontsize=7,
    ncol=2,
)
fig.tight_layout()

## 6b. Winning day: CAS and CAS limit

Actual calibrated airspeed against the CAS limit from `conc_cas_limit.csv`
(altitude x weight -- 530 kt above FL430, lower below). Cruise CAS comes from
the chosen Mach and level; the arrival's decel/level CAS from its Mach, and the
descent flies the 325/350/380 kt CAS schedule directly. The climb has no speed
data.

The descent table holds its schedule CAS all the way down to 1,500 ft, which
is above the CAS limit at low altitude -- a simplification in the table, not
something the notebook adds.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
shade_no_speed_data(ax)
plot_by_phase(ax, "cas_kt", linewidth=2)
ax.plot(
    profile["cum_nm"],
    profile["cas_limit_kt"],
    color="red",
    linestyle="--",
    linewidth=1.3,
)
mark_phase_starts(ax)
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("CAS (kt)")
ax.set_title(f"{title_tag} -- CAS actual vs limit")
ax.set_ylim((0, 600))
ax.legend(
    handles=phase_legend
    + [
        Line2D(
            [0],
            [0],
            color="red",
            linestyle="--",
            linewidth=1.3,
            label="CAS limit",
        )
    ],
    loc="lower right",
    fontsize=7,
    ncol=2,
)
fig.tight_layout()

## 6c. Winning day: Total temperature limits

Cruise only (the march is the only place a temperature is known). Converts actual mach and static temperature at each leg to total (stagnation)
temperature, and compares against the 127°C structural limit. The temperature
margin is highest at lower altitudes where static temp is higher and mach is
lower.

In [ ]:
def total_temperature(mach, static_temp_k):
    """Stagnation (total) temperature given Mach and static temperature (K)."""
    return static_temp_k * (1.0 + 0.2 * mach**2)


total_temp_k = total_temperature(leg["mach"], temp_k_at_best)
total_temp_c = total_temp_k - C_TO_K

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(
    start_cum_nm,
    total_temp_c,
    color=PALETTE[0],
    marker=".",
    markersize=4,
    label="actual total temp",
)
ax.axhline(
    TOTAL_TEMP_MAX_C,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"limit ({TOTAL_TEMP_MAX_C:.0f}°C)",
)
ax.fill_between(
    start_cum_nm, total_temp_c, TOTAL_TEMP_MAX_C, alpha=0.2, color="red"
)
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("total temperature (°C)")
ax.set_title(f"{title_tag} -- total temperature vs limit")
ax.set_ylim((-20, 140))
ax.legend()
fig.tight_layout()

## 7. Winning day: TAS, ground speed, along-track wind

**Along-track wind is only the head/tailwind component**: the wind projected
onto the track (`u*sin(track) + v*cos(track)`), positive = tailwind, negative =
headwind. It is not the full wind speed and it leaves out the crosswind, which
only enters ground speed through the drift correction -- so ground speed is not
exactly TAS + wind.

Dashed lines are means, not curves: the climb has only a whole-climb mean
ground speed and one proxy wind for the whole climb (the table has no
speeds), and the approach allowance only a mean ground speed. Arrival wind is
one value per segment.

In [ ]:
def plot_series(axes, column, color, label):
    """One colour per quantity; dashed where the phase only has a mean."""
    first = True
    for _name, frame in profile.groupby("phase", sort=False):
        mean_only = (frame["speed_basis"] == "mean").all()
        axes.plot(
            frame["cum_nm"],
            frame[column],
            color=color,
            linestyle="--" if mean_only else "-",
            label=label if first else None,
        )
        first = False


fig, ax = plt.subplots(figsize=(11, 4.8))
plot_series(ax, "tas_kt", PALETTE[0], "TAS")
plot_series(ax, "gs_kt", PALETTE[1], "ground speed")
plot_series(ax, "wind_kt", PALETTE[2], "along-track wind (+ tailwind)")
ax.axhline(0.0, color="0.7", linewidth=0.6)
mark_phase_starts(ax)
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("speed (kt)")
ax.set_title(f"{title_tag} -- TAS/GS/wind (dashed = phase mean only)")
ax.legend(fontsize=8)
fig.tight_layout()

## 7a. Winning day: fuel

Everything here comes from the model already run: fuel loaded is TOW minus
ZFW, burn by phase is the climb table, the cruise march and the arrival
segments, and fuel remaining is aircraft weight minus ZFW. Taxi, contingency
and alternate fuel are not modelled -- the flight starts at brake release and
the only reserve is the minimum fuel left at touchdown.

In [ ]:
s = summary
print(
    f"fuel loaded      {s['fuel_loaded_t']:6.1f} t   "
    f"(TOW {s['tow_t']:.1f} t - ZFW {s['zfw_t']:.1f} t)"
)
print(f"  climb          {s['climb_fuel_t']:6.1f} t")
print(f"  cruise         {s['cruise_fuel_t']:6.1f} t")
print(f"  arrival        {s['arrival_fuel_t']:6.1f} t")
print(f"  trip fuel      {s['trip_fuel_t']:6.1f} t")
print(f"landing weight   {s['landing_weight_t']:6.1f} t")
print(
    f"fuel at touchdown{s['landing_fuel_t']:6.1f} t   "
    f"(reserve {s['reserve_t']:.1f} t, "
    f"margin {s['reserve_margin_t']:+.2f} t)"
)
if s["fuel_flag"] or s["arrival_flags"]:
    print(f"!! flags: fuel '{s['fuel_flag']}', arrival '{s['arrival_flags']}'")

fig, (ax_l, ax_r) = plt.subplots(
    1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [3, 2]}
)
plot_by_phase(ax_l, "fuel_remaining_t", linewidth=2)
ax_l.axhline(s["reserve_t"], color="red", linestyle="--", linewidth=1.2)
ax_l.annotate(
    f"reserve {s['reserve_t']:.0f} t",
    xy=(0, s["reserve_t"]),
    xytext=(4, 4),
    textcoords="offset points",
    color="red",
    fontsize=8,
)
mark_phase_starts(ax_l)
ax_l.set_xlabel("distance along route (nm)")
ax_l.set_ylabel("fuel remaining (t)")
ax_l.set_title("Fuel remaining")
ax_l.set_ylim(bottom=0)

bars = ax_r.barh(
    flown["phase"],
    flown["fuel_t"],
    color=[PHASE_COLORS[p] for p in flown["phase"]],
)
ax_r.bar_label(
    bars, labels=[f"{v:.1f} t" for v in flown["fuel_t"]], padding=3, fontsize=8
)
ax_r.invert_yaxis()
ax_r.set_xlabel("fuel burned (t)")
ax_r.set_title(f"Burn by phase (total {flown['fuel_t'].sum():.1f} t)")
ax_r.set_xlim(right=flown["fuel_t"].max() * 1.18)
fig.suptitle(f"{title_tag} -- fuel loaded {s['fuel_loaded_t']:.1f} t")
fig.tight_layout()

### Fuel loaded across all candidates

From the search CSV's own `tow_t` and `zfw_t` (the per-phase split is not in
the CSV). Rows the fixed point flagged `fuel_not_converged` are still plotted.

In [ ]:
fuel_loaded = df["tow_t"] - df["zfw_t"]
winner_fuel = winner_tow_t - ZFW_T

fig, (ax_h, ax_s) = plt.subplots(1, 2, figsize=(12, 4.3))
ax_h.hist(fuel_loaded, bins=60, color=PALETTE[0], alpha=0.8)
ax_h.axvline(
    winner_fuel,
    color=PALETTE[1],
    linestyle="--",
    linewidth=1.2,
    label=f"winner {winner_fuel:.1f} t",
)
ax_h.set_xlabel("fuel loaded (t)")
ax_h.set_ylabel("candidates")
ax_h.set_title("Fuel loaded, all candidates")
ax_h.legend()

ax_s.scatter(
    fuel_loaded,
    df["total_time_s"] / 60.0,
    s=6,
    alpha=0.4,
    color=PALETTE[0],
    edgecolors="none",
)
ax_s.scatter(
    [winner_fuel],
    [winner["total_time_s"] / 60.0],
    color=PALETTE[1],
    s=40,
    zorder=3,
    label="winner",
)
ax_s.set_xlabel("fuel loaded (t)")
ax_s.set_ylabel("total block time (min)")
ax_s.set_title("Fuel loaded vs block time")
ax_s.legend()
fig.tight_layout()

## 7b. Winning day: phases of flight

One row per phase, brake release to touchdown. **climb** is brake release to
LINND (the end of the subsonic limit), **acceleration** is LINND to top of
climb (FL502), **cruise** is top of climb to BARIX, then the arrival model's
**deceleration**, **subsonic cruise** (level, M0.95), **descent** and
**approach**. Takeoff roll and landing are not modelled; the runway penalties
are the time the search adds for wind-unfavourable runways. Means are
time-weighted (Mach/TAS/CAS), distance-weighted (wind) or distance / time
(ground speed); blank where the phase has no such data.

In [ ]:
def _fl_span(entry):
    """'FL0 -> FL407' for one phase row; blank without an altitude."""
    if pd.isna(entry["fl_start"]):
        return ""
    return f"FL{entry['fl_start']:.0f} -> FL{entry['fl_end']:.0f}"


total_row = phases.iloc[-1]
phase_display = pd.DataFrame(
    {
        "phase": phases["phase"],
        "start_utc": phases["start_utc"].dt.strftime("%H:%M:%S"),
        "duration": phases["duration_s"].map(_format_mmss),
        "% time": (
            phases["duration_s"] / total_row["duration_s"] * 100.0
        ).round(1),
        "dist_nm": phases["dist_nm"].round(0),
        "fuel_t": phases["fuel_t"].round(2),
        "altitude": phases.apply(_fl_span, axis=1),
        "mach": phases["mean_mach"].round(2),
        "cas_kt": phases["mean_cas_kt"].round(0),
        "tas_kt": phases["mean_tas_kt"].round(0),
        "gs_kt": phases["mean_gs_kt"].round(0),
        "wind_kt": phases["mean_wind_kt"].round(0),
    }
).fillna("")
display(phase_display)

## 7c. Winning day: route map

The route coloured by phase, over the same phase colours as every plot
above, with the altitude profile underneath. No coastlines (no `cartopy`
dependency) -- just latitude/longitude and the `.pln` waypoints.

In [ ]:
wps = plan["waypoints"]  # (id, lat, lon)
wp_lat = np.array([w[1] for w in wps])
wp_lon = np.array([w[2] for w in wps])

fig, (ax_map, ax_alt) = plt.subplots(
    2, 1, figsize=(11, 7.5), gridspec_kw={"height_ratios": [2.4, 1]}
)
for phase_key, g in profile.groupby("phase", sort=False):
    phase = str(phase_key)
    row = flown[flown["phase"] == phase].iloc[0]
    ax_map.plot(
        g["lon"],
        g["lat"],
        color=PHASE_COLORS[phase],
        linewidth=3.5,
        label=(
            f"{phase}: {_format_mmss(row['duration_s'])}, "
            f"{row['dist_nm']:.0f} nm"
        ),
    )
for phase_start in flown.itertuples():
    p0 = profile[profile["phase"] == phase_start.phase].iloc[0]
    ax_map.plot(
        p0["lon"], p0["lat"], marker="o", color="black", markersize=4, zorder=4
    )
ax_map.scatter(
    wp_lon,
    wp_lat,
    marker="D",
    s=22,
    facecolor="white",
    edgecolor="black",
    zorder=5,
)
for wp_id, lat, lon in wps:
    ax_map.annotate(
        wp_id,
        xy=(lon, lat),
        xytext=(3, 6),
        textcoords="offset points",
        fontsize=7,
    )
ax_map.set_aspect(1.0 / np.cos(np.radians(wp_lat.mean())))
ax_map.grid(color="0.9", linewidth=0.6)
ax_map.set_xlabel("longitude")
ax_map.set_ylabel("latitude")
ax_map.set_title(
    f"{title_tag} -- route by phase "
    "(dots = phase starts, diamonds = waypoints)"
)
ax_map.legend(fontsize=8, loc="upper left")

plot_by_phase(ax_alt, "fl", linewidth=2)
mark_phase_starts(ax_alt)
ax_alt.set_xlabel("distance along route (nm)")
ax_alt.set_ylabel("flight level")
fig.tight_layout()

## 8. Summary

In [ ]:
n = len(df)
jfk_unflyable = int((df["jfk_flag"] == "unflyable").sum())
lhr_unflyable = int((df["lhr_flag"] == "unflyable").sum())
jfk_flagged = int((df["jfk_flag"] == "xwind_25_30").sum())
lhr_flagged = int((df["lhr_flag"] == "xwind_25_30").sum())
lhr_westerly = int((df["lhr_runway"] == "27R/27L").sum())
lhr_easterly = int((df["lhr_runway"] == "09L/09R").sum())

print(
    f"Unflyable: JFK {jfk_unflyable:,} ({jfk_unflyable / n * 100:.1f}%), "
    f"LHR {lhr_unflyable:,} ({lhr_unflyable / n * 100:.1f}%) "
    f"of {n:,} candidates"
)
print(
    f"Flagged {runways.XWIND_OK_KT:.0f}-{runways.XWIND_FLAG_KT:.0f} kt "
    "crosswind-gust band: "
    f"JFK {jfk_flagged:,} ({jfk_flagged / n * 100:.1f}%), "
    f"LHR {lhr_flagged:,} ({lhr_flagged / n * 100:.1f}%)"
)
print(
    f"LHR runway use: westerly (27R/27L) {lhr_westerly:,} "
    f"({lhr_westerly / n * 100:.0f}%), "
    f"easterly (09L/09R) {lhr_easterly:,} ({lhr_easterly / n * 100:.0f}%)"
)